In [23]:
import html2text

def get_email_body(email_message):
    """Extrait le corps texte d'un email, en convertissant le HTML si nécessaire."""
    h = html2text.HTML2Text()
    h.ignore_links = False  # Conserve les liens (modifiable)
    h.ignore_images = True  # Ignore les images
    body = ""

    if email_message.is_multipart():
        for part in email_message.walk():
            content_type = part.get_content_type()
            try:
                if content_type == "text/plain":
                    # Texte brut : priorité maximale
                    body = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                    break
                elif content_type == "text/html" and not body:
                    # HTML : convertir en texte brut
                    html = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                    body = h.handle(html)
                elif "attachment" in str(part.get("Content-Disposition")):
                    # Pièce jointe : optionnel (décommenter pour sauvegarder)
                    filename = part.get_filename()
                    if filename:
                        print(f"  [Pièce jointe: {filename}]")
                        # with open(filename, "wb") as f:
                        #     f.write(part.get_payload(decode=True))
            except Exception as e:
                print(f"  [Erreur lors du traitement d'une partie: {e}]")
                continue
    else:
        # Email simple (non multipart)
        try:
            payload = email_message.get_payload(decode=True)
            charset = email_message.get_content_charset() or "utf-8"
            body = payload.decode(charset, errors="ignore")
        except Exception as e:
            body = f"[Erreur de décodage: {e}]"

    return body.strip()

In [52]:
import os
import imapclient
import email
from email.header import decode_header
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path.home() / '.email_amu.env')
# Charger les variables d'environnement

# load_dotenv('~/.email_amu.env')
HOST = "mail.univ-amu.fr"
USERNAME = "nicolas.catz@univ-amu.fr"
PASSWORD = os.getenv("AMU_EMAIL_PASSWORD")


In [54]:

with imapclient.IMAPClient(HOST, ssl=True) as server:
    server.login(USERNAME, PASSWORD)
    server.select_folder("INBOX", readonly=True)

    messages = server.search("UNSEEN")
    print(f"Nombre d'emails non lus : {len(messages)}")

    for uid, message_data in server.fetch(messages, ["RFC822"]).items():
        email_message = email.message_from_bytes(message_data[b"RFC822"])
        subject = decode_header(email_message["Subject"])[0][0]
        if isinstance(subject, bytes):
            subject = subject.decode("utf-8", errors="ignore")

        print(f"\nUID: {uid}")
        print(f"De: {email_message.get('From', 'Inconnu')}")
        print(f"Sujet: {subject}")

        # Utilisation de la fonction pour extraire le corps
        body = get_email_body(email_message)
        print(f"Corps:\n{body}...")  # Affiche les 500 premiers caractères
        print("-" * 80)

Nombre d'emails non lus : 11

UID: 153649
De: =?UTF-8?q?Chlo=C3=A9=2C_S=C3=A9minaires_Business?= <chloe@seminairesbusiness.com>
Sujet: 2 journée en VIP à Evian Resort !
Corps:
Ce mail provient de l'extérieur, restons vigilants

**Nous vous invitons au plus grand événement annuel des Alpes pour organiser vos séminaires ! **
------------------------------------------------------------------------------------------------

Nous vous invitons au Workshop SBE Montagnes & Lacs à **l’Evian Resort****, dans son hôtel cinq étoiles**, les **28 & 29 mai, l'une des plus belles adresses des Alpes,** face au lac Léman.

**Vous rencontrerez 30 experts du tourismes d'affaires et de l'événementiels sur deux jours :** une sélection d'hôtels, de destinations des Alpes, des incentives et des adresses d’exception, avec qui vous pourrez échanger autour de vos projets. 

**Nous vous offrons le transport, l'hébergement et la restauration**

[ **Je souhaite participer !**](http://r.events.seminairesbusiness.com